# Project 04: QAOA for Max-Cut

This notebook builds a small Max-Cut instance and solves it with QAOA.

In [ ]:
# Imports
import itertools
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

from qiskit.primitives import Sampler
from qiskit_algorithms.minimum_eigensolvers import QAOA
from qiskit_algorithms.optimizers import COBYLA

from qiskit_optimization.applications import Maxcut
from qiskit_optimization.algorithms import MinimumEigenOptimizer

## 1) Generate Graph

In [ ]:
n_nodes = 5
seed = 42
graph = nx.gnp_random_graph(n=n_nodes, p=0.6, seed=seed)

plt.figure(figsize=(5, 4))
nx.draw(graph, with_labels=True, node_color="lightblue", edge_color="gray")
plt.title("Random Graph for Max-Cut")
plt.show()

print("Edges:", list(graph.edges()))

## 2) Build Max-Cut QUBO

In [ ]:
maxcut = Maxcut(graph)
qp = maxcut.to_quadratic_program()
print(qp.export_as_lp_string())

## 3) QAOA Solve (Vary p)

In [ ]:
sampler = Sampler()
depths = [1, 2, 3]
results = []

for p in depths:
    qaoa = QAOA(sampler=sampler, optimizer=COBYLA(maxiter=200), reps=p)
    optimizer = MinimumEigenOptimizer(qaoa)
    res = optimizer.solve(qp)
    cut_value = maxcut.max_cut_value(res.x)
    results.append((p, cut_value, res.x))
    print(f"p={p} | cut={cut_value} | x={res.x}")

## 4) Exact Brute-Force Baseline

In [ ]:
def cut_value_for_bitstring(g, bits):
    value = 0
    for u, v in g.edges():
        if bits[u] != bits[v]:
            value += 1
    return value

best_exact = -1
best_bits = None
for bits in itertools.product([0, 1], repeat=n_nodes):
    value = cut_value_for_bitstring(graph, bits)
    if value > best_exact:
        best_exact = value
        best_bits = bits

print("Exact best cut:", best_exact)
print("Exact best bitstring:", best_bits)

## 5) Plot QAOA Depth vs Cut

In [ ]:
p_vals = [r[0] for r in results]
qaoa_vals = [r[1] for r in results]

plt.figure(figsize=(6, 4))
plt.plot(p_vals, qaoa_vals, marker="o", label="QAOA")
plt.axhline(best_exact, linestyle="--", color="red", label="Exact")
plt.xlabel("QAOA Depth (p)")
plt.ylabel("Cut Value")
plt.title("QAOA Performance vs Depth")
plt.legend()
plt.grid(alpha=0.3)
plt.show()